In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import avg,col,dense_rank



bronze_employees_df=spark.read \
         .option("header","true") \
         .option("inferSchema","true") \
         .csv("/Volumes/workspace/default/first/employees.csv")
display(bronze_employees_df)
bronze_departments_df=spark.read \
           .option("header","true") \
           .option("inferSchema","true") \
           .csv("/Volumes/workspace/default/first/departments.csv")
display(bronze_departments_df)
silver_employees_df=bronze_employees_df
silver_departments_df=bronze_departments_df
silver_employees_df=silver_employees_df.na.drop(subset=["emp_id","dept_id"])
silver_employees_df=silver_employees_df.na.fill({"emp_name":"Unknown","email":"NAN","salary":0})
silver_employees_df=silver_employees_df.dropDuplicates()
silver_departments_df=silver_departments_df.dropDuplicates()
display(silver_employees_df)
display(silver_departments_df)
gold_employees_df=silver_employees_df
gold_departments_df=silver_departments_df
gold_employees_df=gold_employees_df.join(gold_departments_df,"dept_id","left")
display(gold_employees_df)
gold_employees_df=gold_employees_df.orderBy("dept_id")
display(gold_employees_df)
windowSpec=Window.partitionBy("dept_id").orderBy(col("salary").desc())
gold_employees_df=gold_employees_df.withColumn("rank",dense_rank().over(windowSpec))
display(gold_employees_df)








        




emp_id,emp_name,email,dept_id,salary
101,Ram,ram@gmail.com,1,50000
102,null,sita@gmail.com,2,45000
103,Krishna,null,1,60000
104,Ravi,ravi@gmail.com,null,55000
105,Geetha,geetha@gmail.com,2,null
105,Geetha,geetha@gmail.com,2,null
106,Raju,raju@gmail.com,1,62000


dept_id,dept_name
1,IT
2,HR
3,Finance
3,Finance


emp_id,emp_name,email,dept_id,salary
103,Krishna,NAN,1,60000
105,Geetha,geetha@gmail.com,2,0
101,Ram,ram@gmail.com,1,50000
106,Raju,raju@gmail.com,1,62000
102,Unknown,sita@gmail.com,2,45000


dept_id,dept_name
3,Finance
1,IT
2,HR


dept_id,emp_id,emp_name,email,salary,dept_name
1,103,Krishna,NAN,60000,IT
2,105,Geetha,geetha@gmail.com,0,HR
1,101,Ram,ram@gmail.com,50000,IT
1,106,Raju,raju@gmail.com,62000,IT
2,102,Unknown,sita@gmail.com,45000,HR


dept_id,emp_id,emp_name,email,salary,dept_name
1,103,Krishna,NAN,60000,IT
1,101,Ram,ram@gmail.com,50000,IT
1,106,Raju,raju@gmail.com,62000,IT
2,105,Geetha,geetha@gmail.com,0,HR
2,102,Unknown,sita@gmail.com,45000,HR


dept_id,emp_id,emp_name,email,salary,dept_name,rank
1,106,Raju,raju@gmail.com,62000,IT,1
1,103,Krishna,NAN,60000,IT,2
1,101,Ram,ram@gmail.com,50000,IT,3
2,102,Unknown,sita@gmail.com,45000,HR,1
2,105,Geetha,geetha@gmail.com,0,HR,2
